# Agent-to-Agent (A2A) Protocols & Multi-Agent Systems

**Track:** LLMOps & Agentic Systems  
**Toolchain:** CrewAI · AutoGen · Agent Protocol · A2A communication  
**Objective:** Learn how to design and build systems where multiple specialized AI agents communicate, negotiate, and collaborate to solve complex problems.

---

## 🧠 Beyond Single Agents

In the previous notebook (`agentic_orchestration`), we built a single agent that could plan and execute tasks using tools. 

But in the real world, complex problems require **teams** of specialists. 

### 🤔 Why Multi-Agent Systems?

| Scenario | Problem with Single Agent | Multi-Agent Solution |
|----------|-------------------------|----------------------|
| **Software Dev** | One agent writes code, reviews it, and tests it (conflict of interest). | Coder agent writes, Reviewer agent critiques, Tester agent validates. |
| **Context Limit** | One agent gets confused trying to hold the whole system architecture in context. | Specialized agents only see relevant context. |
| **Tool Overload** | Giving one agent 50 tools leads to poor tool selection. | Give 5 tools each to 10 specialized agents. |
| **Cross-Org Apps** | Your company's AI needs to book a flight with Delta's AI. | Standardized Agent-to-Agent (A2A) protocols. |

### The Multi-Agent Topologies

```
1. Hierarchical (Manager-Worker)
   [Manager] → assigns tasks to → [Researcher], [Writer], [Editor]

2. Sequential (Assembly Line)
   [Data Gatherer] → passes data to → [Analyzer] → passes report to → [Summarizer]

3. Joint Chat (War Room / AutoGen style)
   [User], [Coder_Agent], [Critic_Agent] all chat in one group thread.
```

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** A massive universal agent gets easily confused. Seniors split problems into hierarchical teams of micro-agents (Coder, Reviewer, QA) that communicate securely through standardized Agent-to-Agent (A2A) JSON protocols.

**Mental Model:** Multi-Agent systems are corporate departments. Instead of one panicked employee doing everything, the Manager agent decomposes the task and delegates strict sub-tasks to the Research and Dev departments.

**Common Junior Pitfall:** Letting agents converse in unstructured natural language. In production, agents must communicate in strict Pydantic JSON schemas so the logic is deterministic and system boundaries are secure.

---


In [ ]:
# Cell 1 — Install
!pip install -q pydantic

## 1 · The Hierarchical Multi-Agent Pattern

The most reliable production pattern is the **Manager-Worker** topology. The manager plans the execution, delegates tasks to specialists, and synthesizes the final result.

In [ ]:
# Cell 2 — Simulating a multi-agent hierarchy
import time

class Agent:
    def __init__(self, name, role, skills):
        self.name = name
        self.role = role
        self.skills = skills
        
    def execute(self, task, context=None):
        print(f'  🤖 [{self.name} | {self.role}] processing: "{task[:40]}..."')
        time.sleep(0.5)  # Simulate LLM thinking
        
        if self.role == 'Researcher':
            return f"[Research Data: Found 5 competitors for feature X]"
        elif self.role == 'Engineer':
            return f"[Code: Implemented feature X using Python]"
        elif self.role == 'QA':
            if 'Code:' in context:
                return "[Review: Code looks good, added 2 test cases]"
            return "[Review: Cannot test without code]"
        
        return "[Task completed]"

class ManagerAgent:
    def __init__(self, team):
        self.team = {agent.role: agent for agent in team}
        
    def delegate(self, objective):
        print(f'👔 [Manager] Objective: "{objective}"')
        print(f'👔 [Manager] Devising execution plan...')
        
        # The manager explicitly orchestrates the workflow
        results = {}
        
        # Step 1: Research
        res_data = self.team['Researcher'].execute('Analyze competitors for new feature')
        results['research'] = res_data
        
        # Step 2: Engineering (depends on research)
        code_data = self.team['Engineer'].execute('Build feature based on research', context=res_data)
        results['code'] = code_data
        
        # Step 3: QA (depends on code)
        qa_data = self.team['QA'].execute('Review and test the code', context=code_data)
        results['qa'] = qa_data
        
        print(f'👔 [Manager] All tasks complete. Synthesizing final response.')
        return f"Successfully built and tested feature based on competitor analysis."

# Create the team
team = [
    Agent('Alice', 'Researcher', ['web_search', 'pdf_reader']),
    Agent('Bob', 'Engineer', ['python', 'github']),
    Agent('Charlie', 'QA', ['pytest', 'linter'])
]

manager = ManagerAgent(team)

print('🏢 Multi-Agent Hierarchy Execution')
print('=' * 60)
final_result = manager.delegate('Build the new AI chat feature for our app')
print(f'\n✅ Final Output: {final_result}')

print('\n💡 Notice how the Manager coordinates dependencies. The QA agent')
print('   doesn\'t start until the Engineer is finished.')

---

## 2 · Standardizing Agent-to-Agent (A2A) Protocols

### 🤔 The Interoperability Problem

Right now, if your company uses a CrewAI agent, and your vendor uses a LangChain agent, **they cannot easily talk to each other.** 

As AI moves to B2B interactions (e.g., your supply chain agent negotiating with a supplier's inventory agent), we need **standardized A2A protocols**—just like HTTP standardizes web servers.

### Core Requirements of an A2A Protocol

1. **Discovery:** "What capabilities does your agent have?"
2. **Authentication:** "Are you authorized to request this action?"
3. **State Management:** "What is our conversation ID?"
4. **Format:** Standard JSON payload for requests and responses.

### 🌟 The Agent Protocol (Standardization Effort)

An emerging open standard (Agent Protocol) defines a universal API for agents:

```http
POST /ap/v1/agent/tasks
{
  "input": "Book a flight to NYC"
}

POST /ap/v1/agent/tasks/{task_id}/steps
{
  "input": "Proceed with payment"
}
```

In [ ]:
# Cell 3 — Simulating an A2A Handshake
from pydantic import BaseModel
from typing import List, Dict, Optional
import json

# 1. Define the universal A2A payload structure
class AgentMessage(BaseModel):
    sender_id: str
    recipient_id: str
    conversation_id: str
    intent: str  # e.g., 'request_info', 'propose_action', 'confirm'
    payload: dict
    auth_token: str

class SupplyChainAgent:
    """An agent that belongs to the Manufacturer."""
    def __init__(self):
        self.id = "agent_manufacturer_01"
        
    def receive_message(self, msg: AgentMessage):
        print(f'📥 [Manufacturer] Received A2A message from {msg.sender_id}')
        
        if msg.intent == 'check_inventory':
            item = msg.payload.get('item_id')
            print(f'   Looking up inventory for {item}...')
            
            # Respond using the same standard
            return AgentMessage(
                sender_id=self.id,
                recipient_id=msg.sender_id,
                conversation_id=msg.conversation_id,
                intent='inventory_response',
                payload={'item_id': item, 'in_stock': 450, 'price': 12.50},
                auth_token='sys_xyz789'
            )

class ProcurementAgent:
    """An agent that belongs to the Buyer."""
    def __init__(self):
        self.id = "agent_buyer_99"
        
    def check_supplier_stock(self, target_agent_id, item_id):
        print(f'📤 [Buyer] Initiating A2A request to {target_agent_id}')
        request = AgentMessage(
            sender_id=self.id,
            recipient_id=target_agent_id,
            conversation_id="conv_12345",
            intent="check_inventory",
            payload={"item_id": item_id},
            auth_token="sys_abc123"
        )
        return request

# Simulate the interaction across organizational boundaries
buyer = ProcurementAgent()
supplier = SupplyChainAgent()

print('🤝 Cross-Organization A2A Communication')
print('=' * 60)

# 1. Buyer sends formatted A2A payload
outbound_req = buyer.check_supplier_stock(supplier.id, "widget_A")

print(f'\n[Network Payload Transmitted]')
print(json.dumps(outbound_req.model_dump(), indent=2))
print()

# 2. Supplier receives and processes
inbound_resp = supplier.receive_message(outbound_req)

print(f'\n[Network Response Transmitted]')
print(json.dumps(inbound_resp.model_dump(), indent=2))

print('\n💡 Because both agents speak the same Pydantic schema (the A2A protocol),')
print('   they can negotiate B2B transactions without human intervention.')

---

## 3 · Agent Frameworks (2026 Landscape)

If you are building multi-agent systems, you shouldn't write the orchestration logic from scratch. 

| Framework | Paradigm | Best For |
|-----------|----------|----------|
| **LangGraph** | Graph-based state machine | Production reliability, exact control over flow |
| **CrewAI** | Role-based teams | Fast prototyping of hierarchical agent teams |
| **AutoGen** | Conversational agents | Complex, open-ended problem solving via chat |
| **Swarm** (OpenAI) | Lightweight agent handoffs | Simple, stateless agent routing |

### ⚖️ Trade-off: Determinism vs Autonomy

| High Autonomy (AutoGen) | High Determinism (LangGraph) |
|-------------------------|------------------------------|
| Agents figure out how to work together | You define the exact workflow graph |
| Can solve unexpected problems | Solves specific problems reliably |
| Risk of infinite loops or going off-topic | Safe, predictable execution |
| *Best for: Research, complex coding* | *Best for: Production enterprise apps* |

---

## 🎯 Summary & Key Takeaways

- **Multi-Agent Systems (MAS)** separate concerns. Instead of one confused "do-it-all" prompt, you have specialized agents with specific tools and strict roles.
- **A2A Protocols** are the HTTP of the agentic web. They allow agents across different networks/companies to negotiate and share data using standardized JSON schemas.
- **Manager-Worker Topology** is the most reliable structure for production business applications.
- **Framework Choice:** Use CrewAI for fast prototyping, LangGraph for production predictability.

**Next Steps:** Proceed to the next notebooks to understand how these agents are monitored and physically hosted.